### Ingest races.csv

In [0]:
%run /Workspace/Users/nicolas97alonso@gmail.com/databricks-course/utils/vairables

### Step 1 - Read races csv

#### 1.1 Difine Schema 

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType

In [0]:
races_schema = StructType(
    [
        StructField("raceId", IntegerType(), False),
        StructField("year", IntegerType(), True),
        StructField("round", IntegerType(), True),
        StructField("circuitId", IntegerType(), True),
        StructField("name", StringType(), True),
        StructField("date", DateType(), True),
        StructField("time", StringType(), True),
        StructField("url", StringType(), True),
    ]
)

#### 1.2 Read the csv 

In [0]:
raw_path = define_path("raw")

In [0]:
races_df = (
    spark.read
        .option("header", True)
        .schema(races_schema)
        .csv(f"{raw_path}/races.csv")
)

### Step 2 - Add / Transform columns

In [0]:
from pyspark.sql.functions import to_timestamp, concat, lit, current_timestamp

In [0]:
transformed_circuits_df = (
    races_df
        .withColumn(
            "race_timestamp",
            to_timestamp(
                concat(col("date"), lit(" "), col("time")),
                "yyyy-MM-dd HH:mm:ss"
            )
        )
        .withColumn("ingestion_date", current_timestamp())
)

### Step 3 - Select Columns

In [0]:
from pyspark.sql.functions import col

In [0]:
races_selected_df = transformed_circuits_df.select(
    col("raceId"),
    col("year"),
    col("round"),
    col("circuitId"),
    col("name"),
    col("race_timestamp"),
    col("ingestion_date")
)

### Step 4 - Rename Columns

In [0]:
reces_renamed_df = (
    races_selected_df 
    .withColumnRenamed("raceId", "race_id")
    .withColumnRenamed("year","race_year")
    .withColumnRenamed("circuitId","circuit_id")
)

Step 5 - Write data to parquet

In [0]:
processed_path = define_path("processed")

In [0]:
reces_renamed_df.write.mode("overwrite").parquet(f"{processed_path}/races")

In [0]:
spark.read.parquet(f"{processed_path}/races").display()